# Model Evaluation

## TGB Models

### Load the Dataset and Split

In [27]:
import pandas as pd
df_test = pd.read_parquet('../Data/TGB Preprocessed/df_test.parquet')

print("Shape: ", df_test.shape)

for col in df_test.columns:
    print (f"{col}: ", df_test[col].dtype)

Shape:  (199848, 483)
id:  category
name:  category
country:  category
latitude:  float32
longitude:  float32
minBorderDistanceKm:  float32
minCapitalDistanceKm:  float32
periodStart:  datetime64[ns]
month_sin:  float32
month_cos:  float32
is_voting:  bool
TargetConflictLocal_t+1:  boolean
TargetConflictRegional_t+1:  boolean
TargetConflictLocal_t+3:  boolean
TargetConflictRegional_t+3:  boolean
TargetConflictLocal_t+6:  boolean
TargetConflictRegional_t+6:  boolean
TargetConflictLocal_t+12:  boolean
TargetConflictRegional_t+12:  boolean
LocalEventCount_std2:  float32
LocalEventCount_std3:  float32
LocalEventCount_std6:  float32
LocalEventCount_std12:  float32
LocalEventCount_std18:  float32
LocalDistinctActorCount_std2:  float32
LocalDistinctActorCount_std3:  float32
LocalDistinctActorCount_std6:  float32
LocalDistinctActorCount_std12:  float32
LocalDistinctActorCount_std18:  float32
LocalDistinctEventTypes_std2:  float32
LocalDistinctEventTypes_std3:  float32
LocalDistinctEventTypes_s

### Load TGB Models from File and Evaluate per Target

In [28]:
#define target columns, we'll iterate over this list later. These need to be exact matches to stop target information leaking into the training (looking at you, AUC = 1) 
target_columns = [
"TargetConflictLocal_t+1",
"TargetConflictRegional_t+1",
"TargetConflictLocal_t+3",
"TargetConflictRegional_t+3",
"TargetConflictLocal_t+6",
"TargetConflictRegional_t+6",
"TargetConflictLocal_t+12",
"TargetConflictRegional_t+12"
]

In [29]:
def get_top_features(model, feature_cols, name, top_n=20):
    """Return a DataFrame with top-N features for a given model."""
    fi = pd.DataFrame({
        "feature": feature_cols,
        name: model.feature_importances_
    }).sort_values(name, ascending=False).head(top_n)
    return fi

In [30]:
import matplotlib.pyplot as plt

def table_to_image(headers, rows, filename=f"../Evaluation/model_comparison.png"):
    fig, ax = plt.subplots(figsize=(10, 2 + len(rows)*0.4))
    ax.axis('off')

    # Create table
    table = plt.table(
        cellText=rows,
        colLabels=headers,
        loc='center',
        cellLoc='center'
    )

    # Style
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 1.5)

    # Save as image
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close(fig)

In [35]:
import joblib
import gc 

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Model Evaluation:")

for target_column in target_columns:

    print(f"\nEvaluating model for: {target_column}")

    #build testing dataset (y; actual target values)
    #test_ids = subsamples_test[target_column]
    #df_test = #df[df["id"].isin(test_ids)].copy()

    y_test = df_test[[target_column]].copy()

    #build the testing dataset (x):
    feature_cols = [
        c for c in df_test.columns
        if c not in target_columns
        and c not in ["id", "placeName", "name", "year", 'latitude', 'longitude', "periodStart", "periodEnd", "period_midpoint"]
    ]

    X_test = df_test[feature_cols].copy()
    
    print("X_test shape:", X_test.shape)
    print("y_test positives:", y_test[target_column].sum())

    #obtain model from dict
    #model = models[target_column]

    #load model from disk (pickle files are neat, this saves tedious mucking around with only saving part of the model):
    model_lgbm = joblib.load(f"../Models/LightGBM/lightgbm_model_{target_column}.pkl")
    model_rf = joblib.load(f"../Models/RF/rf_model_{target_column}.pkl")
    model_catboost = joblib.load(f"../Models/CatBoost/catboost_model_{target_column}.pkl")
    
    #predict
    y_pred_prob_lgbm = model_lgbm.predict_proba(X_test)[:, 1]
    y_pred_lgbm = (y_pred_prob_lgbm >= 0.5).astype(int)

    y_pred_prob_rf = model_rf.predict_proba(X_test)[:, 1]
    y_pred_rf = (y_pred_prob_rf >= 0.5).astype(int)

    y_pred_prob_catboost = model_catboost.predict_proba(X_test)[:, 1]
    y_pred_catboost = (y_pred_prob_catboost >= 0.5).astype(int)

    #Conf matrix
    print("Confusion Matricies")
    print("LightGBM")
    cm_lgbm = confusion_matrix(y_test, y_pred_lgbm) #note this all breaks for regional_12, as there are 0 positive instances. 
    print(cm_lgbm)
    tn_lgbm, fp_lgbm, fn_lgbm, tp_lgbm = cm_lgbm.ravel()


    #specificity = TN / (TN + FP)
    specificity_lgbm = round(tn_lgbm / (tn_lgbm + fp_lgbm) if (tn_lgbm + fp_lgbm) > 0 else 0.0, 2)

    #Classification metrics:
    precision_lgbm  = round(precision_score(y_test, y_pred_lgbm , zero_division=0), 2)
    recall_lgbm  = round(recall_score(y_test, y_pred_lgbm , zero_division=0), 2)
    f1_lgbm  = round(f1_score(y_test, y_pred_lgbm , zero_division=0), 2)
    auc_lgbm  = round(roc_auc_score(y_test, y_pred_prob_lgbm) if y_test[target_column].nunique() > 1 else float("nan"), 2)

    print("\nClassification Report (LightGBM):")
    print(classification_report(y_test, y_pred_lgbm, zero_division=0))

    print("Random Forest")
    cm_rf = confusion_matrix(y_test, y_pred_rf) #note this all breaks for regional_12, as there are 0 positive instances. 
    print(cm_rf)
    tn_rf, fp_rf, fn_rf, tp_rf = cm_rf.ravel()

    #specificity = TN / (TN + FP)
    specificity_rf = tn_rf / (tn_rf + fp_rf) if (tn_rf + fp_rf) > 0 else 0.0

    #Classification metrics:
    accuracy_rf = round(accuracy_score(y_test, y_pred_rf ), 2)
    precision_rf  = round(precision_score(y_test, y_pred_rf , zero_division=0), 2)
    recall_rf  = round(recall_score(y_test, y_pred_rf , zero_division=0), 2)
    f1_rf  = round(f1_score(y_test, y_pred_rf , zero_division=0), 2)
    auc_rf  = round(roc_auc_score(y_test, y_pred_prob_rf) if y_test[target_column].nunique() > 1 else float("nan"), 2)

    print("\nClassification Report (Random Forest):")
    print(classification_report(y_test, y_pred_rf, zero_division=0))
    
    print("CatBoost")
    cm_catboost = confusion_matrix(y_test, y_pred_catboost) #note this all breaks for regional_12, as there are 0 positive instances. 
    print(cm_catboost)
    
    tn_catboost, fp_catboost, fn_catboost, tp_catboost = cm_catboost.ravel()

    #specificity = TN / (TN + FP)
    specificity_catboost = tn_catboost / (tn_catboost + fp_catboost) if (tn_catboost + fp_catboost) > 0 else 0.0

    #Classification metrics:
    accuracy_catboost = round(accuracy_score(y_test, y_pred_catboost ), 2)
    precision_catboost  = round(precision_score(y_test, y_pred_catboost , zero_division=0), 2)
    recall_catboost  = round(recall_score(y_test, y_pred_catboost , zero_division=0), 2)
    f1_catboost  = round(f1_score(y_test, y_pred_catboost , zero_division=0), 2)
    auc_catboost  = round(roc_auc_score(y_test, y_pred_prob_catboost) if y_test[target_column].nunique() > 1 else float("nan"), 2)

    print("\nClassification Report (CatBoost):")
    print(classification_report(y_test, y_pred_catboost, zero_division=0))

    print("\nCombined Classification Metrics")
    #
    #headers = ["Model", "Accuracy", "Specivity", "Precision", "Recall", "F1", "AUC"]
    #rows = [
    #    ["LightGBM", {accuracy_lgbm},{specificity_lgbm}, {precision_lgbm}, {recall_lgbm}, {f1_lgbm}, {auc_lgbm}],
    #    ["Random Forest", {accuracy_rf},{specificity_rf}, {precision_rf}, {recall_rf}, {f1_rf}, {auc_rf}],
    #    ["CatBoost", {accuracy_catboost},{specificity_catboost}, {precision_catboost}, {recall_catboost}, {f1_catboost}, {auc_catboost}]
    #]
    

    results_df = pd.DataFrame([
        {
            "Model": "LightGBM",
            "Accuracy": round(accuracy_lgbm, 2),
            "Specificity": round(specificity_lgbm, 2),
            "Precision": round(precision_lgbm, 2),
            "Recall": round(recall_lgbm, 2),
            "F1": round(f1_lgbm, 2),
            "AUC": round(auc_lgbm, 2)
        },
        {
            "Model": "Random Forest",
            "Accuracy": round(accuracy_rf, 2),
            "Specificity": round(specificity_rf, 2),
            "Precision": round(precision_rf, 2),
            "Recall": round(recall_rf, 2),
            "F1": round(f1_rf, 2),
            "AUC": round(auc_rf, 2)
        },
        {
            "Model": "CatBoost",
            "Accuracy": round(accuracy_catboost, 2),
            "Specificity": round(specificity_catboost, 2),
            "Precision": round(precision_catboost, 2),
            "Recall": round(recall_catboost, 2),
            "F1": round(f1_catboost, 2),
            "AUC": round(auc_catboost, 2)
        }
    ])
    
    results_df.to_csv(f"../Evaluation/model_metrics_{target_column}.csv", index=False)
    print (results_df)

    print("\nFeature Importance")
    
    # Extract top 20 for each model
    fi_lgbm = get_top_features(model_lgbm, feature_cols, "lgbm_importance")
    fi_rf   = get_top_features(model_rf,   feature_cols, "rf_importance")
    fi_cat  = get_top_features(model_catboost, feature_cols, "cat_importance")
     
     # Merge side-by-side on feature name
    fi_merged = (
         fi_lgbm
         .merge(fi_rf, on="feature", how="outer")
         .merge(fi_cat, on="feature", how="outer")
    )

    fi_merged.to_csv(f"../Evaluation/feature_importance_{target_column}.csv", index=False)
    
    print(fi_merged)
   
    gc.collect()

    print(f"End of evaluation for model {target_column}")
    print("------")
    #end of this iteration
    

Model Evaluation:

Evaluating model for: TargetConflictLocal_t+1
X_test shape: (199848, 470)
y_test positives: 18815
Confusion Matricies
LightGBM
[[180352    681]
 [     6  18809]]

Classification Report (LightGBM):
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00    181033
         1.0       0.97      1.00      0.98     18815

    accuracy                           1.00    199848
   macro avg       0.98      1.00      0.99    199848
weighted avg       1.00      1.00      1.00    199848

Random Forest
[[ 17499 163534]
 [   140  18675]]

Classification Report (Random Forest):
              precision    recall  f1-score   support

         0.0       0.99      0.10      0.18    181033
         1.0       0.10      0.99      0.19     18815

    accuracy                           0.18    199848
   macro avg       0.55      0.54      0.18    199848
weighted avg       0.91      0.18      0.18    199848

CatBoost
[[180609    424]
 [     1  18814